In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

path = '/home/z5562390/work/dataset2023/dataset1.csv.gz'
output_path = 'MathV2_Top3&5.csv' # Renamed for differentiation

# ==========================================
# Core Prediction Engine: Extract Top-5 Historical Patterns
# ==========================================
def process_single_user(user_df):
    uid = user_df['uid'].iloc[0]
    
    # 1. Prepare training data
    train_df = user_df[user_df['d'] <= 59].drop_duplicates(subset=['d', 't'], keep='last')
    if train_df.empty: return pd.DataFrame()

    # 2. Build full grid and apply Forward Fill
    full_index = pd.MultiIndex.from_product([[uid], range(60), range(48)], names=['uid', 'd', 't'])
    train_full = train_df.set_index(['uid', 'd', 't']).reindex(full_index)
    train_full = train_full.ffill().bfill().reset_index()
    train_full['weekday'] = train_full['d'] % 7
    
    # --- NEW: Extract top 5 global locations as fallback ---
    global_counts = train_full.groupby(['x', 'y']).size().reset_index(name='count').sort_values('count', ascending=False)
    global_locs = list(zip(global_counts['x'], global_counts['y']))
    while len(global_locs) < 5: 
        global_locs.append(global_locs[0] if global_locs else (0,0))
    
    # 3. Core calculation: Find all visited locations sorted by frequency for each state
    loc_counts = train_full.groupby(['weekday', 't', 'x', 'y']).size().reset_index(name='count')
    sorted_locs = loc_counts.sort_values(by=['weekday', 't', 'count'], ascending=[True, True, False])
    
    # Aggregate coordinates into lists: [(x1,y1), (x2,y2)...]
    wt_agg = sorted_locs.groupby(['weekday', 't']).apply(
        lambda df: list(zip(df['x'], df['y'])), include_groups=False
    ).reset_index(name='loc_list')

    # 4. Generate prediction grid for Days 60-74
    pred_index = pd.MultiIndex.from_product([[uid], range(60, 75), range(48)], names=['uid', 'd', 't'])
    pred_df = pd.DataFrame(index=pred_index).reset_index()
    pred_df['weekday'] = pred_df['d'] % 7

    # 5. Merge predictions
    ans = pd.merge(pred_df, wt_agg, on=['weekday', 't'], how='left')
    
    # --- NEW: Dynamically extract Top-5 and handle edge cases ---
    def extract_top5(loc_list):
        if not isinstance(loc_list, list): return global_locs[:5]
        
        res = []
        seen = set()
        # Priority 1: Use historical locations for current (weekday, time)
        for loc in loc_list:
            if loc not in seen:
                seen.add(loc); res.append(loc)
            if len(res) == 5: break
                
        # Priority 2: Fallback to global top locations if < 5
        for loc in global_locs:
            if len(res) == 5: break
            if loc not in seen:
                seen.add(loc); res.append(loc)
                
        # Final fallback for extreme edge cases
        while len(res) < 5: res.append(res[0])
        return res

    ans['top5'] = ans['loc_list'].apply(extract_top5)
    
    # Unpack list into separate columns x1, y1 ... x5, y5
    for i in range(5):
        ans[f'x{i+1}'] = ans['top5'].apply(lambda item: item[i][0])
        ans[f'y{i+1}'] = ans['top5'].apply(lambda item: item[i][1])
    
    # Clean up and return target columns in order
    output_cols = ['uid', 'd', 't', 'x1', 'y1', 'x2', 'y2', 'x3', 'y3', 'x4', 'y4', 'x5', 'y5']
    return ans[output_cols].astype(int)

# ==========================================
# Execution Engine: Chunk Reading & Buffer Stitching
# ==========================================
output_cols = ['uid', 'd', 't', 'x1', 'y1', 'x2', 'y2', 'x3', 'y3', 'x4', 'y4', 'x5', 'y5']
pd.DataFrame(columns=output_cols).to_csv(output_path, index=False)
dtypes = {'uid': np.int32, 'd': np.int16, 't': np.int8, 'x': np.int16, 'y': np.int16}
buffer_df = pd.DataFrame()

TARGET_USERS = 10000 
user_count, stop_reading = 0, False

print(f"Starting Top-5 calculation for {TARGET_USERS} users...")

for chunk in pd.read_csv(path, usecols=['uid', 'd', 't', 'x', 'y'], dtype=dtypes, chunksize=500000):
    if stop_reading: break
        
    chunk = pd.concat([buffer_df, chunk], ignore_index=True)
    unique_uids = chunk['uid'].unique()
    
    if len(unique_uids) == 1:
        buffer_df = chunk; continue
        
    last_uid = unique_uids[-1]
    complete_data, buffer_df = chunk[chunk['uid'] != last_uid], chunk[chunk['uid'] == last_uid]
    
    for uid, user_data in complete_data.groupby('uid'):
        if user_count >= TARGET_USERS: stop_reading = True; break
            
        user_pred = process_single_user(user_data)
        if not user_pred.empty:
            user_pred.to_csv(output_path, mode='a', header=False, index=False)
            
        user_count += 1
        if user_count % 1000 == 0: print(f"Processed {user_count} / {TARGET_USERS} users...")

if not stop_reading and not buffer_df.empty and user_count < TARGET_USERS:
    for uid, user_data in buffer_df.groupby('uid'):
        user_pred = process_single_user(user_data)
        if not user_pred.empty: user_pred.to_csv(output_path, mode='a', header=False, index=False)
        user_count += 1

print(f"\nTask Complete! Results saved to: {output_path}")

Starting Top-5 calculation for 10000 users...
Processed 1000 / 10000 users...
Processed 2000 / 10000 users...
Processed 3000 / 10000 users...
Processed 4000 / 10000 users...
Processed 5000 / 10000 users...
Processed 6000 / 10000 users...
Processed 7000 / 10000 users...
Processed 8000 / 10000 users...
Processed 9000 / 10000 users...
Processed 10000 / 10000 users...

Task Complete! Results saved to: MathV2_Top3&5.csv


In [ ]:
pred_path = 'MathV2_Top3&5.csv' 
orig_path = '/home/z5562390/work/dataset2023/dataset1.csv.gz'

print("1. Loading prediction file...")
pred_df = pd.read_csv(pred_path)
target_uids = pred_df['uid'].unique()
print(f"-> Successfully loaded predictions for {len(target_uids)} users.")

print("\n2. Extracting Ground Truth from original dataset (Days 60-74)...")
gt_chunks = []
dtypes = {'uid': np.int32, 'd': np.int16, 't': np.int8, 'x': np.int16, 'y': np.int16}

# Stream reading to prevent memory overflow
for chunk in pd.read_csv(orig_path, usecols=['uid', 'd', 't', 'x', 'y'], dtype=dtypes, chunksize=1000000):
    # Filter for days 60-74 and targeted users only
    chunk = chunk[(chunk['d'] >= 60) & (chunk['d'] <= 74)]
    chunk = chunk[chunk['uid'].isin(target_uids)]
    chunk = chunk.drop_duplicates(subset=['uid', 'd', 't'], keep='last')
    
    if not chunk.empty:
        gt_chunks.append(chunk)

gt_df = pd.concat(gt_chunks, ignore_index=True)
print(f"-> Successfully extracted {len(gt_df)} true trajectory records.")

print("\n3. Calculating Accuracies...")
# Merge ground truth with predictions (GT cols: x, y; Pred cols: x1...y5)
eval_df = pd.merge(gt_df, pred_df, on=['uid', 'd', 't'])

# Check if candidate locations 1 to 5 match GT
match_1 = (eval_df['x'] == eval_df['x1']) & (eval_df['y'] == eval_df['y1'])
match_2 = (eval_df['x'] == eval_df['x2']) & (eval_df['y'] == eval_df['y2'])
match_3 = (eval_df['x'] == eval_df['x3']) & (eval_df['y'] == eval_df['y3'])
match_4 = (eval_df['x'] == eval_df['x4']) & (eval_df['y'] == eval_df['y4'])
match_5 = (eval_df['x'] == eval_df['x5']) & (eval_df['y'] == eval_df['y5'])

# Calculate cumulative hits (Logical OR: |)
eval_df['is_correct_top1'] = match_1
eval_df['is_correct_top3'] = match_1 | match_2 | match_3
eval_df['is_correct_top5'] = match_1 | match_2 | match_3 | match_4 | match_5

# Calculate accuracy percentages
acc_top1 = eval_df['is_correct_top1'].mean() * 100
acc_top3 = eval_df['is_correct_top3'].mean() * 100
acc_top5 = eval_df['is_correct_top5'].mean() * 100

print("==================================================")
print("🏆 COMPREHENSIVE EVALUATION RESULTS")
print("==================================================")
print(f"Total GT records evaluated: {len(eval_df)}")
print("-" * 50)
print(f"🥇 Top-1 Hits: {eval_df['is_correct_top1'].sum():>8}  (Accuracy: {acc_top1:.2f} %)")
print(f"🥈 Top-3 Hits: {eval_df['is_correct_top3'].sum():>8}  (Accuracy: {acc_top3:.2f} %)")
print(f"🥉 Top-5 Hits: {eval_df['is_correct_top5'].sum():>8}  (Accuracy: {acc_top5:.2f} %)")
print("==================================================")

1. 正在加载预测文件...
-> 成功加载 10000 个用户的预测数据。

2. 正在从原数据集提取 Ground Truth (第 60-74 天)...
-> 成功提取 2470704 条真实的轨迹变动记录。

3. 正在计算各项 Accuracy...
🏆 COMPREHENSIVE EVALUATION RESULTS
总评估真实记录数: 2470704
--------------------------------------------------
🥇 Top-1 命中次数:   611679  (准确率: 24.76 %)
🥈 Top-3 命中次数:   978804  (准确率: 39.62 %)
🥉 Top-5 命中次数:  1156144  (准确率: 46.79 %)
